In [1]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

# Enable mixed precision
mixed_precision.set_global_policy('float32')

print("Compute dtype:", mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", mixed_precision.global_policy().variable_dtype)

Compute dtype: float32
Variable dtype: float32


In [2]:
from datasets import load_dataset

ds = load_dataset("Yelp/yelp_review_full")

README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [4]:
import pandas as pd

train_df=pd.DataFrame(ds["train"])
train_df

,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...
...,...,...
649995,4,I had a sprinkler that was gushing... pipe bro...
649996,0,Phone calls always go to voicemail and message...
649997,0,Looks like all of the good reviews have gone t...
649998,4,I was able to once again rely on Yelp to provi...


In [5]:
train_df["label"].value_counts()

label
4    130000
1    130000
3    130000
0    130000
2    130000
Name: count, dtype: int64

In [6]:
gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus)

strategy = tf.distribute.MirroredStrategy()

print("Number of replicas:", strategy.num_replicas_in_sync)

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of replicas: 2


In [7]:
new_train=pd.DataFrame()

for i in train_df["label"].unique():
    temp=train_df[train_df["label"]==i].sample(frac=0.01,random_state=42)
    new_train=pd.concat([new_train,temp],axis=0)

In [8]:
new_train["label"].value_counts()

label
4    1300
1    1300
3    1300
0    1300
2    1300
Name: count, dtype: int64

In [9]:
new_train=new_train[["text"]].reset_index(drop=True)
new_train

,text
0,This place is good. I prefer to buy my DVDs fr...
1,I've made several trips to Indonesia over the ...
2,This is a perfectly awesome sports bar.\n\nYou...
3,"I don't do caffeine anymore, or calories for t..."
4,Cafe Davio...ahhhhhhh. I have bben delighted ...
...,...
6495,I got a slice of pepperoni for $3.75 (I think)...
6496,Great place to go for experimental cinema. For...
6497,2.5 stars. The Luxor is the epitome of medioc...
6498,Such a great name - Perfect Pear Bistro! Servi...


In [10]:
import re

def text_preprocessing(text):
    text=text.lower()
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)
    text=re.sub(r"\\n","",text)
    text=re.sub(r"[^a-z0-9\s.,?!']", " ", text).strip()
    text=re.sub(r"\s+"," ",text)
    return text

In [11]:
new_train["text"]=new_train["text"].apply(text_preprocessing)
new_train

,text
0,this place is good. i prefer to buy my dvds fr...
1,i've made several trips to indonesia over the ...
2,this is a perfectly awesome sports bar.you can...
3,"i don't do caffeine anymore, or calories for t..."
4,cafe davio...ahhhhhhh. i have bben delighted t...
...,...
6495,i got a slice of pepperoni for 3.75 i think . ...
6496,great place to go for experimental cinema. for...
6497,2.5 stars. the luxor is the epitome of mediocr...
6498,such a great name perfect pear bistro! service...


In [12]:
new_train=new_train["text"].values.tolist()
new_train[:10]

['this place is good. i prefer to buy my dvds from here than to rent because i prefer to call dvds i have my own. some of the best exrentals i have bought out of here. freahy friday among others. if you want cheap dvds, cheap rental or 25 dvds per week sent to your door for just u00a310 pop in here.',
 "i've made several trips to indonesia over the years and this is the first place in the states that serves food comparable to the fare i had there.. the food at bandung is excellent and the service is first rate. regrettably, i find myself in madison only two or three times a year. otherwise, i'd be a very regular customer.",
 "this is a perfectly awesome sports bar.you can go there for any big game for any sport, and they will have it on one of their many screens with sound!! they even have a roll down projection screen towards the rear of the bar that shows the big games as well.i recently went to picassos one week night, and ordered a beer and an order of a dozen cajun garlic wings w 

In [13]:
len(new_train)

6500

In [14]:
import re

def split_sentences(text):
    setentences=re.split(r"(?<=[.!?])\s+",text)
    return [s.strip() for s in setentences if s.strip()]

In [15]:
clean_train=[]
for x in new_train:
    l=split_sentences(x)
    clean_train.extend(l)

In [16]:
len(clean_train),len(new_train)

(57829, 6500)

In [17]:
from tensorflow.keras.preprocessing.text import Tokenizer

tk = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

In [18]:
tk.fit_on_texts(clean_train)

In [ ]:
tk.index_word

In [19]:
len(tk.index_word)

26475

In [20]:
tk.texts_to_sequences(["how are you"])

[[122, 29, 21]]

In [21]:
samp=tk.texts_to_sequences([new_train[0]])[0]
samp

[17,
 31,
 11,
 30,
 4,
 1255,
 6,
 496,
 14,
 5581,
 51,
 43,
 90,
 6,
 1921,
 72,
 4,
 1255,
 6,
 387,
 5581,
 4,
 24,
 14,
 400,
 63,
 8,
 2,
 112,
 1,
 4,
 24,
 864,
 39,
 8,
 43,
 1,
 713,
 2626,
 620,
 37,
 21,
 134,
 406,
 5581,
 406,
 1347,
 48,
 789,
 5581,
 732,
 444,
 1025,
 6,
 65,
 414,
 10,
 36,
 7538,
 1306,
 12,
 43]

In [22]:
from sklearn.model_selection import train_test_split

train_docs,val_docs=train_test_split(clean_train,test_size=0.3,random_state=32)

In [23]:
len(train_docs),len(val_docs)

(40480, 17349)

In [24]:
total_tokens = 0
oov_tokens = 0

for text in val_docs:
    seq = tk.texts_to_sequences([text])[0]

    total_tokens += len(seq)
    oov_tokens += seq.count(tk.word_index["<OOV>"])

oov_rate = oov_tokens / total_tokens * 100

print(f"OOV Rate: {oov_rate:.2f}%")

OOV Rate: 2.43%


In [25]:
MAX_LEN = 100
import numpy as np

def generate_sequence(documents, min_words=2):

    X, Y = [], []

    for line in documents:
        tk_list = tk.texts_to_sequences([line])[0]

        for i in range(min_words, len(tk_list)):
            context = tk_list[max(0, i-MAX_LEN):i]

            x = np.zeros(MAX_LEN, dtype=np.int32)
            x[:len(context)] = context

            X.append(x)
            Y.append(tk_list[i])

    return np.asarray(X, dtype=np.int32), np.asarray(Y, dtype=np.int32)

In [26]:
x_train,y_train=generate_sequence(train_docs)
x_val,y_val=generate_sequence(val_docs)

In [27]:
x_train.shape,y_train.shape

((540302, 100), (540302,))

In [28]:
x_val.shape,y_val.shape

((230455, 100), (230455,))

In [29]:
x_train

array([[  9,   7,   0, ...,   0,   0,   0],
       [  9,   7,  38, ...,   0,   0,   0],
       [  9,   7,  38, ...,   0,   0,   0],
       ...,
       [276,   5, 445, ...,   0,   0,   0],
       [276,   5, 445, ...,   0,   0,   0],
       [276,   5, 445, ...,   0,   0,   0]], dtype=int32)

In [30]:
import numpy as np

np.save("yelp-train-features.npy",x_train)
np.save("yelp-train-target.npy",y_train)
np.save("yelp-val-features.npy",x_val)
np.save("yelp-val-target.npy",y_val)

import joblib

joblib.dump(tk,"yelp-tokenizer.pkl")

['yelp-tokenizer.pkl']

In [31]:
import tensorflow as tf
BATCH_SIZE=1024
AUTOTUNE=tf.data.AUTOTUNE


train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_ds = (
    train_ds
    .shuffle(10000)
    .batch(BATCH_SIZE, drop_remainder=False)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [32]:
vocab_size=tk.num_words
vocab_size

10000

In [33]:
import gensim.downloader as api

api.info()["models"].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [34]:
from gensim.models import KeyedVectors

glove = KeyedVectors.load(
    "/kaggle/input/datasets/abhinaykalavakuri/glove-100/glove_twitter_100.kv",
    mmap="r"
)

In [35]:
vocab_size = tk.num_words

embeddings_matrix = np.zeros((vocab_size, 100))
c = 0

for i in range(1, vocab_size):
    word = tk.index_word[i]

    if word in glove.index_to_key:
        embeddings_matrix[i, :] = glove[word]
    else:
        embeddings_matrix[i, :] = np.random.normal(0, 0.05, 100)
        c += 1

print("Missing GloVe embeddings:", c)
print("Embedding matrix shape:", embeddings_matrix.shape)

Missing GloVe embeddings: 554
Embedding matrix shape: (10000, 100)


In [36]:
import numpy as np

np.save("yelp-embeddings-10k.npy",embeddings_matrix)


import joblib

joblib.dump(tk,"yelp-tokenizer-10k.pkl")

['yelp-tokenizer-10k.pkl']

In [43]:
from tensorflow.keras.layers import Embedding,Input,Dense,GRU,Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2

In [44]:
vocab_size

10000

In [47]:
with strategy.scope():
    gru_model2=Sequential()
    gru_model2.add(Input(shape=(100,)))
    gru_model2.add(Embedding(vocab_size,100,weights=[embeddings_matrix],trainable=True,mask_zero=True))
    gru_model2.add(GRU(units=384,return_sequences=True,dropout=0.15,kernel_regularizer=l2(1e-5)))
    gru_model2.add(GRU(units=256,return_sequences=True,dropout=0.15,kernel_regularizer=l2(1e-5)))
    gru_model2.add(GRU(units=128,return_sequences=False,kernel_regularizer=l2(1e-6)))
    gru_model2.add(Dense(128,activation="relu",kernel_regularizer=l2(1e-6)))
    gru_model2.add(Dropout(0.2))
    gru_model2.add(Dense(vocab_size,activation="softmax"))
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003,clipnorm=1.0)
    gru_model2.compile(optimizer=optimizer,loss="sparse_categorical_crossentropy",metrics=['accuracy',tf.keras.metrics.SparseTopKCategoricalAccuracy(
        k=5, name="top_5_accuracy")])

In [48]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    mode="min",
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_top_5_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "nwp_chk_10k_100d_384neuron_gru.keras",
    monitor="val_top_5_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

gru_model2.fit(train_ds,epochs=200,callbacks = [
    checkpoint,
    reduce_lr,
    early_stop
],validation_data=val_ds)

Epoch 1/200
INFO:tensorflow:Collective all_reduce tensors: 13 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


INFO:tensorflow:Collective all_reduce tensors: 13 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


INFO:tensorflow:Collective all_reduce IndexedSlices: 1 all_reduces, num_devices =2, group_size = 2, implementation = CommunicationImplementation.NCCL


INFO:tensorflow:Collective all_reduce IndexedSlices: 1 all_reduces, num_devices =2, group_size = 2, implementation = CommunicationImplementation.NCCL


527/528 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.0354 - loss: 7.1148 - top_5_accuracy: 0.1278
Epoch 1: val_top_5_accuracy improved from None to 0.16334, saving model to nwp_chk_10k_100d_384neuron_gru.keras

Epoch 1: finished saving model to nwp_chk_10k_100d_384neuron_gru.keras
528/528 ━━━━━━━━━━━━━━━━━━━━ 48s 83ms/step - accuracy: 0.0436 - loss: 6.7203 - top_5_accuracy: 0.1523 - val_accuracy: 0.0476 - val_loss: 6.5557 - val_top_5_accuracy: 0.1633 - learning_rate: 3.0000e-04
Epoch 2/200
527/528 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.0476 - loss: 6.5603 - top_5_accuracy: 0.1634
Epoch 2: val_top_5_accuracy did not improve from 0.16334
528/528 ━━━━━━━━━━━━━━━━━━━━ 42s 79ms/step - accuracy: 0.0478 - loss: 6.5581 - top_5_accuracy: 0.1631 - val_accuracy: 0.0476 - val_loss: 6.5536 - val_top_5_accuracy: 0.1633 - learning_rate: 3.0000e-04
Epoch 3/200
527/528 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.0479 - loss: 6.5565 - top_5_accuracy: 0.1637
Epoch 3: val_top_5_accuracy di

### Testing

In [51]:
test_df=pd.DataFrame(ds["test"])
test_df

,label,text
0,0,I got 'new' tires from them and within two wee...
1,0,Don't waste your time. We had two different p...
2,0,All I can say is the worst! We were the only 2...
3,0,I have been to this restaurant twice and was d...
4,0,Food was NOT GOOD at all! My husband & I ate h...
...,...,...
49995,0,Just wanted to write a review to chip in with ...
49996,4,Great ambience. Great drinks. Great food. I lo...
49997,3,I have been to the other Monks locations so I ...
49998,1,Don't go here. I know you might want to try i...


In [52]:
test_df["text"]=test_df["text"].apply(text_preprocessing)
test_df=test_df[["text"]]
test_df

,text
0,i got 'new' tires from them and within two wee...
1,don't waste your time. we had two different pe...
2,all i can say is the worst! we were the only 2...
3,i have been to this restaurant twice and was d...
4,food was not good at all! my husband i ate her...
...,...
49995,just wanted to write a review to chip in with ...
49996,great ambience. great drinks. great food. i lo...
49997,i have been to the other monks locations so i ...
49998,don't go here. i know you might want to try it...


In [53]:
test_df=test_df["text"].values.tolist()
test_df[:10]

["i got 'new' tires from them and within two weeks got a flat. i took my car to a local mechanic to see if i could get the hole patched, but they said the reason i had a flat was because the previous patch had blown wait, what? i just got the tire and never needed to have it patched? this was supposed to be a new tire. i took the tire over to flynn's and they told me that someone punctured my tire, then tried to patch it. so there are resentful tire slashers? i find that very unlikely. after arguing with the guy and telling him that his logic was far fetched he said he'd give me a new tire this time . i will never go back to flynn's b c of the way this guy treated me and the simple fact that they gave me a used tire!",
 "don't waste your time. we had two different people come to our house to give us estimates for a deck one of them the owner . both times, we never heard from them. not a call, not the estimate, nothing.",
 'all i can say is the worst! we were the only 2 people in the pl

In [54]:
clean_test=[]
for x in test_df:
    l=split_sentences(x)
    clean_test.extend(l)
clean_test[:5]

["i got 'new' tires from them and within two weeks got a flat.",
 'i took my car to a local mechanic to see if i could get the hole patched, but they said the reason i had a flat was because the previous patch had blown wait, what?',
 'i just got the tire and never needed to have it patched?',
 'this was supposed to be a new tire.',
 "i took the tire over to flynn's and they told me that someone punctured my tire, then tried to patch it."]

In [55]:
x_test,y_test=generate_sequence(clean_test)

In [57]:
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [58]:
test_results=gru_model2.evaluate(test_ds,return_dict=True,verbose=1)
test_results

5787/5787 ━━━━━━━━━━━━━━━━━━━━ 154s 26ms/step - accuracy: 0.1849 - loss: 5.0610 - top_5_accuracy: 0.3827


{'accuracy': 0.1849393993616104,
 'loss': 5.060971736907959,
 'top_5_accuracy': 0.38273337483406067}